# Mg₃Bi₂ — analysis & figure generation

Generates every figure in this notebook straight into
`Mg3Bi2-Mechanical-MLIAP-Dataset/figures/<category>/`, so `figures/` in the
dataset repo is finally populated instead of empty.

**Run this with the `snap` kernel** (Kernel → Change Kernel → Python (jup_snap)).
That conda env already has `gpyumd` and `phonopy` installed; the base
`anaconda3` env does not.

## Data provenance — read this before trusting a number

Not every plot below is backed by a raw simulation file on this machine.
Three markers are used throughout:

- **[reproducible]** — loads real output files (`.out`, `.dat`, `.yaml`,
  `model.xyz`, ...) from `~/Mg3Bi2/<category>/`. Re-running the cell
  regenerates the figure from source.
- **[transcribed — no source file found]** — the numbers are hardcoded
  arrays in the cell. They came from a real prior calculation (phono3py,
  a Wigner-transport run, or an HNEMD run at pressures this machine no
  longer has raw output for), but there is no `.out`/`.dat` file anywhere
  under `~/Mg3Bi2/` or in this repo to regenerate them from. Treat these as
  frozen results, not as something this notebook can verify or update.
- **[removed]** — see the note at the very end of this notebook for what
  was dropped from the original `Mg3Bi2.ipynb` and why.

## Known data gaps (found while auditing input paths)

- `~/Mg3Bi2/hnemd/` only has pressure **0 GPa** raw HNEMD/SHC data. The
  1-9 GPa tensor-kappa figure has no matching raw directory on this
  machine — it's a [transcribed] case.
- `~/Mg3Bi2/hnemd/0/600/Z/shc.out` is **missing** (`kappamode.out` is
  present). The 600 K / Z-direction spectral (HNEMA+SHC) plot silently
  skips its SHC overlay for that one case — not a bug, just a gap in the
  raw output.
- No phono3py or Wigner-transport (WTE, "smm19-RTA") output directory
  exists anywhere on this machine — both thermal-conductivity methods are
  [transcribed] only.
- `~/Mg3Bi2/dispersion/` has **two different NEP phonon-dispersion
  results**: `band.yaml` and `band_nep.yaml` (different content, both
  distinct from the DFT `band_dft.yaml`). This notebook uses `band.yaml`
  to match what the original notebook did — confirm that's the NEP run
  you want reported; swap the filename in the phonon-dispersion cell if
  `band_nep.yaml` is the current/correct one.

## Setup

One shared style block instead of the ~30 copy-pasted versions in the
original notebook, and one `savefig_fig()` helper that always writes into
the right `figures/<category>/` subfolder as both PDF and PNG.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, AutoMinorLocator, NullLocator
from matplotlib.lines import Line2D

REPO_FIGURES = "/home/ashwani/Mg3Bi2/Mg3Bi2-Mechanical-MLIAP-Dataset/figures"
REPO_ROOT = "/home/ashwani/Mg3Bi2/Mg3Bi2-Mechanical-MLIAP-Dataset"
SIBLING = "/home/ashwani/Mg3Bi2"   # raw simulation outputs live here, not in the repo

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 18,
    "axes.labelsize": 18,
    "axes.titlesize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 15,
    "axes.linewidth": 2.0,
    "lines.linewidth": 2.5,
    "lines.markersize": 8,
})

def apply_style(ax):
    ax.tick_params(axis='both', which='major', direction='inout', length=8, width=2)
    ax.tick_params(axis='both', which='minor', direction='in', length=5, width=1.5)
    ax.minorticks_on()
    ax.xaxis.set_major_locator(MaxNLocator(6))
    ax.yaxis.set_major_locator(MaxNLocator(6))
    ax.set_aspect("auto")

def savefig_fig(fig, category, name, **kwargs):
    # Save fig as both .pdf and .png into figures/<category>/ inside the repo.
    out_dir = os.path.join(REPO_FIGURES, category)
    os.makedirs(out_dir, exist_ok=True)
    kwargs.setdefault("bbox_inches", "tight")
    fig.savefig(os.path.join(out_dir, f"{name}.pdf"), **kwargs)
    fig.savefig(os.path.join(out_dir, f"{name}.png"), dpi=300, **kwargs)
    print(f"saved {category}/{name}.[pdf|png]")

## Elastic constants vs. temperature  **[reproducible]**

Source: `~/Mg3Bi2/elastic_data/Mg3Bi2_elastic_tensor_vs_T.dat`
(columns: `T C11 C22 C33 C12 C13 C23 C44 C55 C66 C14 ...`).
Derives VRH bulk/shear moduli, Young's modulus, and Poisson's ratio, and
fits everything linearly in `T` up to 800 K.

In [ ]:
data = np.loadtxt(os.path.join(SIBLING, "elastic_data", "Mg3Bi2_elastic_tensor_vs_T.dat"))
data = data[data[:, 0] <= 800.0]

T = data[:, 0]
C11, C22, C33 = data[:, 1], data[:, 2], data[:, 3]
C12, C13, C23 = data[:, 4], data[:, 5], data[:, 6]
C44, C55, C66 = data[:, 7], data[:, 8], data[:, 9]
C14 = data[:, 10]
CIJ = {"C11": C11, "C12": C12, "C13": C13, "C33": C33, "C44": C44, "C14": C14}

def linear_fit(x, y):
    m, c = np.polyfit(x, y, 1)
    return m, c, m * x + c

cij_fits = {k: linear_fit(T, v) for k, v in CIJ.items()}

B_V, G_V, B_R, G_R = [], [], [], []
for i in range(len(T)):
    Cm = np.array([
        [C11[i], C12[i], C13[i], 0, 0, 0],
        [C12[i], C22[i], C23[i], 0, 0, 0],
        [C13[i], C23[i], C33[i], 0, 0, 0],
        [0, 0, 0, C44[i], 0, 0],
        [0, 0, 0, 0, C55[i], 0],
        [0, 0, 0, 0, 0, C66[i]],
    ])
    Bv = (C11[i] + C22[i] + C33[i] + 2 * (C12[i] + C13[i] + C23[i])) / 9.0
    Gv = ((C11[i] + C22[i] + C33[i] - C12[i] - C13[i] - C23[i])
          + 3 * (C44[i] + C55[i] + C66[i])) / 15.0
    S = np.linalg.inv(Cm)
    Br = 1.0 / (S[0, 0] + S[1, 1] + S[2, 2] + 2 * (S[0, 1] + S[0, 2] + S[1, 2]))
    Gr = 15.0 / (4 * (S[0, 0] + S[1, 1] + S[2, 2])
                 - 4 * (S[0, 1] + S[0, 2] + S[1, 2])
                 + 3 * (S[3, 3] + S[4, 4] + S[5, 5]))
    B_V.append(Bv); G_V.append(Gv); B_R.append(Br); G_R.append(Gr)

B = 0.5 * (np.array(B_V) + np.array(B_R))
G = 0.5 * (np.array(G_V) + np.array(G_R))
E = (9.0 * B * G) / (3.0 * B + G)
nu = (3.0 * B - 2.0 * G) / (2.0 * (3.0 * B + G))
BG = B / G

mB, cB, B_fit = linear_fit(T, B)
mG, cG, G_fit = linear_fit(T, G)
mBG, cBG, BG_fit = linear_fit(T, BG)
mE, cE, E_fit = linear_fit(T, E)
mnu, cnu, nu_fit = linear_fit(T, nu)

print("Linear temperature coefficients (T <= 800 K):")
for k, (m, c, _) in cij_fits.items():
    print(f"  {k:<6} dC/dT = {m: .4e} GPa/K")
print(f"  {'B_VRH':<6} dB/dT = {mB: .4e} GPa/K")
print(f"  {'G_VRH':<6} dG/dT = {mG: .4e} GPa/K")
print(f"  {'B/G':<6} d(B/G)/dT = {mBG: .4e} 1/K")
print(f"  {'E_VRH':<6} dE/dT = {mE: .4e} GPa/K")
print(f"  {'nu_VRH':<6} dnu/dT = {mnu: .4e} 1/K")

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
styles = {"C11": ("o", "C0"), "C12": ("s", "C1"), "C13": ("^", "C2"),
          "C33": ("o", "C3"), "C44": ("s", "C4"), "C14": ("D", "C5")}
for name, (marker, color) in styles.items():
    m, c, fit_line = cij_fits[name]
    ax.plot(T, fit_line, linestyle=(0, (6, 3)), color=color, linewidth=3.0, alpha=0.9, zorder=1)
    ax.plot(T, CIJ[name], marker=marker, linestyle="None", color=color, zorder=3)
ax.set_xlabel("Temperature (T) [K]")
ax.set_ylabel(r"Elastic constants" + "\n" + r"($C_{ij}$) [GPa]")
legend_handles = [Line2D([0], [0], marker=m, linestyle=(0, (6, 3)), color=c,
                          linewidth=2.5, markersize=8, label=rf"$C_{{{name[1:]}}}$")
                  for name, (m, c) in styles.items()]
ax.legend(handles=legend_handles, frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, 0.80))
apply_style(ax)
fig.tight_layout()
savefig_fig(fig, "elastic_constants", "elastic_constants_linear_fit_vs_T")
plt.show()
plt.close(fig)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(T, E, "o", color="purple")
ax.plot(T, E_fit, "--", color="purple")
ax.set_xlabel("Temperature [K]")
ax.set_ylabel("Young's modulus [GPa]")
ax.legend(handles=[Line2D([0], [0], color="purple", marker="o", linestyle="--", label=r"$E_{\mathrm{VRH}}$")],
          frameon=False, loc="upper right")
apply_style(ax)
fig.tight_layout()
savefig_fig(fig, "elastic_constants", "youngs_modulus_vs_T")
plt.show()
plt.close(fig)

In [ ]:
fig, ax1 = plt.subplots(figsize=(5, 4))
ax1.plot(T, B, "o", color="#1f77b4", label=r"$B_{\mathrm{VRH}}$")
ax1.plot(T, B_fit, "--", color="#1f77b4")
ax1.plot(T, G, "s", color="#ff7f0e", label=r"$G_{\mathrm{VRH}}$")
ax1.plot(T, G_fit, "--", color="#ff7f0e")
ax1.set_xlabel("Temperature [K]")
ax1.set_ylabel("Modulus [GPa]")
apply_style(ax1)

ax2 = ax1.twinx()
ax2.plot(T, BG, "^", color="black", label=r"$B/G$")
ax2.plot(T, BG_fit, "--", color="black")
ax2.set_ylabel(r"Pugh ratio $B/G$")
ax2.yaxis.set_major_locator(MaxNLocator(6))
ax2.tick_params(axis="y", which="major", direction="inout", length=7, width=2)
ax2.minorticks_on()

legend_handles = [
    Line2D([0], [0], color="#1f77b4", marker="o", linestyle="--", label=r"$B_{\mathrm{VRH}}$"),
    Line2D([0], [0], color="#ff7f0e", marker="s", linestyle="--", label=r"$G_{\mathrm{VRH}}$"),
    Line2D([0], [0], color="black", marker="^", linestyle="--", label=r"$B/G$"),
]
ax1.legend(handles=legend_handles, frameon=False, loc="center left", bbox_to_anchor=(-0.05, 0.65))
fig.tight_layout()
savefig_fig(fig, "elastic_constants", "bulk_shear_modulus_and_BG_ratio_vs_T")
plt.show()
plt.close(fig)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(T, nu, "o", color="darkgreen", label=r"$\nu_{\mathrm{VRH}}$")
ax.plot(T, nu_fit, "--", color="darkgreen")
ax.set_xlabel("Temperature [K]")
ax.set_ylabel(r"Poisson's ratio $\nu$")
apply_style(ax)
ax.legend(handles=[Line2D([0], [0], color="darkgreen", marker="o", linestyle="--", label=r"$\nu_{\mathrm{VRH}}$")],
          frameon=False, loc="best")
fig.tight_layout()
savefig_fig(fig, "elastic_constants", "poisson_ratio_vs_T")
plt.show()
plt.close(fig)

## Thermal expansion  **[reproducible]**

Source: `~/Mg3Bi2/thermal_expansion/thermo_{P}GPa.out` (GPUMD `run.in`
thermo output) for P = 0,1,3,5,7,9 GPa, T = 100-1000 K in 10 blocks.
Requires `gpyumd` (`snap` kernel).

In [ ]:
from gpyumd.load import load_thermo

dt = 0.01
NT = 10
temp = np.arange(100, 1001, 100)
A0_PRIM, C0_PRIM = 4.6043, 7.2764   # DFT primitive cell (Ang), used to detect the supercell size

pressure_cases = [
    ("0 GPa", "thermo_0GPa.out"), ("1 GPa", "thermo_1GPa.out"), ("3 GPa", "thermo_3GPa.out"),
    ("5 GPa", "thermo_5GPa.out"), ("7 GPa", "thermo_7GPa.out"), ("9 GPa", "thermo_9GPa.out"),
]
te_base_dir = os.path.join(SIBLING, "thermal_expansion")

te_results = {}
for label, fname in pressure_cases:
    thermo = load_thermo(directory=te_base_dir, filename=fname)
    La = np.sqrt(thermo['ax']**2 + thermo['ay']**2 + thermo['az']**2)
    Lb = np.sqrt(thermo['bx']**2 + thermo['by']**2 + thermo['bz']**2)
    Lc = np.sqrt(thermo['cx']**2 + thermo['cy']**2 + thermo['cz']**2)
    NXY = int(round(La[0] / A0_PRIM))
    NZ = int(round(Lc[0] / C0_PRIM))
    a_lat = (La + Lb) / (2 * NXY)
    c_lat = Lc / NZ
    Pave = (thermo['Px'] + thermo['Py'] + thermo['Pz']) / 3.0

    nsteps = len(a_lat)
    nsteps_eff = (nsteps // NT) * NT
    time = dt * np.arange(1, nsteps_eff + 1)
    a_lat, c_lat, Pave = a_lat[:nsteps_eff], c_lat[:nsteps_eff], Pave[:nsteps_eff]
    Tvec = thermo['temperature'][:nsteps_eff]

    M = nsteps_eff // NT
    a_ave = a_lat.reshape(NT, M)[:, M // 2:].mean(axis=1)
    c_ave = c_lat.reshape(NT, M)[:, M // 2:].mean(axis=1)
    V_ave = np.sqrt(3) / 2 * a_ave**2 * c_ave

    fit_a = np.poly1d(np.polyfit(temp, a_ave, 1))
    fit_c = np.poly1d(np.polyfit(temp, c_ave, 1))
    alpha_a = fit_a.c[0] / np.mean(a_ave)
    alpha_c = fit_c.c[0] / np.mean(c_ave)
    alpha_V = 2.0 * alpha_a + alpha_c   # hexagonal: V ~ a^2 c

    te_results[label] = dict(time=time, temperature=Tvec, pressure=Pave, a_lat=a_lat, c_lat=c_lat,
                              a_ave=a_ave, c_ave=c_ave, V_ave=V_ave, fit_a=fit_a, fit_c=fit_c,
                              alpha_a=alpha_a, alpha_c=alpha_c, alpha_V=alpha_V, NXY=NXY, NZ=NZ)

print("Thermal expansion coefficients (100-1000 K):")
print(f"{'P':>8} {'alpha_a (1/K)':>16} {'alpha_c (1/K)':>16} {'alpha_V (1/K)':>16}")
for label, _ in pressure_cases:
    d = te_results[label]
    print(f"{label:>8} {d['alpha_a']:>16.3e} {d['alpha_c']:>16.3e} {d['alpha_V']:>16.3e}")

In [ ]:
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
legend_handles = [Line2D([0], [0], color=colors[i], marker='o', markerfacecolor='none',
                          linewidth=2.5, markersize=8, label=label)
                   for i, (label, _) in enumerate(pressure_cases)]

fig, ax = plt.subplots(figsize=(5, 4))
for label, d in te_results.items():
    ax.plot(d["time"], d["temperature"], label=label)
ax.set_xlabel("Time (ps)"); ax.set_ylabel("Temperature (K)")
ax.legend(frameon=False); apply_style(ax); fig.tight_layout()
savefig_fig(fig, "thermal_expansion", "temperature_vs_time_all_pressures")
plt.show(); plt.close(fig)

fig, ax = plt.subplots(figsize=(5, 4))
for label, d in te_results.items():
    ax.plot(d["time"], d["pressure"], label=label)
ax.set_xlabel("Time (ps)"); ax.set_ylabel("Pressure (GPa)")
ax.legend(frameon=False); apply_style(ax); fig.tight_layout()
savefig_fig(fig, "thermal_expansion", "pressure_vs_time_all_pressures")
plt.show(); plt.close(fig)

fig, ax = plt.subplots(figsize=(5, 5.4))
for i, (label, _) in enumerate(pressure_cases):
    d = te_results[label]
    ax.scatter(temp, d["a_ave"], facecolors="none", edgecolors=colors[i])
    ax.plot([0, 1100], d["fit_a"]([0, 1100]), color=colors[i])
ax.set_xlabel("Temperature [K]"); ax.set_ylabel(r"$a$ [$\AA$]")
ax.legend(handles=legend_handles, frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.4), ncol=2)
apply_style(ax); fig.tight_layout()
savefig_fig(fig, "thermal_expansion", "a_vs_T_all_pressures")
plt.show(); plt.close(fig)

fig, ax = plt.subplots(figsize=(5, 5.4))
for i, (label, _) in enumerate(pressure_cases):
    d = te_results[label]
    ax.scatter(temp, d["c_ave"], facecolors="none", edgecolors=colors[i])
    ax.plot([0, 1100], d["fit_c"]([0, 1100]), color=colors[i])
ax.set_xlabel("Temperature [K]"); ax.set_ylabel(r"$c$ [$\AA$]")
ax.legend(handles=legend_handles, frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.4), ncol=2)
apply_style(ax); fig.tight_layout()
savefig_fig(fig, "thermal_expansion", "c_vs_T_all_pressures")
plt.show(); plt.close(fig)

fig, ax = plt.subplots(figsize=(5, 5.4))
for i, (label, _) in enumerate(pressure_cases):
    d = te_results[label]
    ax.scatter(temp, d["V_ave"], facecolors="none", edgecolors=colors[i])
ax.set_xlabel("Temperature [K]"); ax.set_ylabel(r"$V$ [$\AA^3$]")
ax.legend(handles=legend_handles, frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.4), ncol=2)
apply_style(ax); fig.tight_layout()
savefig_fig(fig, "thermal_expansion", "V_vs_T_all_pressures")
plt.show(); plt.close(fig)

## Melting point from MSD  **[reproducible]**

Source: `~/Mg3Bi2/melting/out_{P}GPa` (heating-ramp MD, column 2 =
temperature/time axis, column 10 = MSD) for P = 0,1,3,5,7,9 GPa.

Two independent detectors are combined: a sustained-MSD-above-threshold
test and a normalized-slope test on the smoothed MSD. Their average is
reported as $T_m$, half their difference as the uncertainty — this
replaces the original notebook's practice of hand-copying detector output
into a separate hardcoded array for the summary plot.

In [ ]:
PRESSURES = [0, 1, 3, 5, 7, 9]
melt_files = {P: os.path.join(SIBLING, "melting", f"out_{P}GPa") for P in PRESSURES}
x_col, msd_col = 2, 10

def load_msd(fname):
    return np.loadtxt(fname, comments="#", skiprows=1)

def detect_melting_msd_threshold(x, msd, msd_threshold=5.0, sustain_window=20):
    for i in range(len(msd) - sustain_window):
        if np.all(msd[i:i + sustain_window] > msd_threshold):
            return x[i]
    return None

def detect_melting_slope(x, msd, smooth_window=15, slope_threshold=0.2, sustain=10):
    msd_s = np.convolve(msd, np.ones(smooth_window) / smooth_window, mode="same")
    slope = np.gradient(msd_s, x)
    slope_norm = slope / np.percentile(slope, 95)
    for i in range(len(slope_norm) - sustain):
        if x[i] < 0.6 * np.max(x):
            continue
        if np.all(slope_norm[i:i + sustain] > slope_threshold):
            return x[i]
    return None

melt_results = {}
fig, ax = plt.subplots(figsize=(7, 5))
for P in PRESSURES:
    data = load_msd(melt_files[P])
    x, msd = data[:, x_col], data[:, msd_col]
    Tm_msd = detect_melting_msd_threshold(x, msd)
    Tm_slope = detect_melting_slope(x, msd)
    if Tm_msd is not None and Tm_slope is not None:
        Tm_final, Tm_err = 0.5 * (Tm_msd + Tm_slope), 0.5 * abs(Tm_msd - Tm_slope)
    else:
        Tm_final, Tm_err = (Tm_msd or Tm_slope), None
    melt_results[P] = dict(Tm_msd=Tm_msd, Tm_slope=Tm_slope, Tm_final=Tm_final, Tm_err=Tm_err)
    ax.plot(x, msd, lw=2, label=f"{P} GPa")
    if Tm_final is not None:
        ax.axvline(Tm_final, ls="--", alpha=0.4)

ax.set_xlabel("Temperature / Time"); ax.set_ylabel(r"MSD ($\mathrm{\AA}^2$)")
ax.legend(frameon=False); ax.grid(alpha=0.3); apply_style(ax); fig.tight_layout()
savefig_fig(fig, "melting_point", "msd_vs_temperature_all_pressures")
plt.show(); plt.close(fig)

print("P (GPa) | MSD-based | slope-based | final Tm | uncertainty")
for P in PRESSURES:
    d = melt_results[P]
    print(f"{P:>6}  | {str(d['Tm_msd']):>9} | {str(d['Tm_slope']):>11} | "
          f"{str(d['Tm_final']):>8} | {d['Tm_err']}")

Tm_vals = [melt_results[P]['Tm_final'] for P in PRESSURES if melt_results[P]['Tm_final'] is not None]
tol = 20.0
if all((y - x) > -tol for x, y in zip(Tm_vals, Tm_vals[1:])):
    print("\nMelting temperature increases with pressure (physically consistent).")
else:
    print("\nNon-monotonic behavior detected -- check equilibration or heating rate.")

In [ ]:
P_arr = np.array(PRESSURES, dtype=float)
Tm_computed = np.array([melt_results[P]['Tm_final'] for P in PRESSURES], dtype=float)

# Previously curated values from a more careful manual pass over the same MSD curves
# (kept for comparison -- see the markdown note above on why this isn't blindly trusted).
Tm_curated = np.array([1028.676, 1086.5177, 1268.27095, 1275.18095, 1412.147, 1541.26985])

print(f"{'P (GPa)':>8} {'Tm auto-detect (K)':>20} {'Tm curated (K)':>16} {'diff (K)':>10}")
for p, ta, tc in zip(P_arr, Tm_computed, Tm_curated):
    print(f"{p:>8.0f} {ta:>20.1f} {tc:>16.1f} {ta - tc:>10.1f}")

coef_lin = np.polyfit(P_arr, Tm_curated, 1)
Tm_lin = np.polyval(coef_lin, P_arr)
slope, intercept = coef_lin

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(P_arr, Tm_curated, s=70, color="tab:blue", zorder=3, label="_nolegend_")
ax.plot(P_arr, Tm_lin, linestyle="--", color="tab:blue", linewidth=2.5, label="_nolegend_")
ax.plot([], [], marker='o', linestyle='--', color='tab:blue', linewidth=2.5, markersize=8, label="$T_m$ (MD)")
ax.scatter(P_arr, Tm_computed, s=70, marker='x', color="tab:red", zorder=4, label="$T_m$ (this notebook, auto-detect)")
ax.text(0.3, 0.20, rf"$T_m(P) = {slope:.1f}\,P + {intercept:.0f}$", transform=ax.transAxes,
        fontsize=15, va="top", ha="left", bbox=dict(facecolor="white", edgecolor="none", alpha=0.8))
ax.set_xlabel("Pressure (P) [GPa]"); ax.set_ylabel("Melting Temperature\n($T_m$) [K]")
ax.set_xlim(-0.5, 10); apply_style(ax); ax.legend(frameon=False); ax.grid(alpha=0.3)
fig.tight_layout()
savefig_fig(fig, "melting_point", "melting_temperature_vs_pressure")
plt.show(); plt.close(fig)

## Radial distribution function — solid vs. liquid  **[reproducible]**

Source: `~/Mg3Bi2/rdf/rdf_{300,1100}K.out` (columns: r, total, Mg-Mg,
Bi-Bi, Mg-Bi). `adf_{300,1100}K.out` also exist in that folder but are not
used by any figure here or in the original notebook -- angular
distribution functions, left for a future addition if wanted.

In [ ]:
rdf_dir = os.path.join(SIBLING, "rdf")
d300 = np.loadtxt(os.path.join(rdf_dir, "rdf_300K.out"))
d1100 = np.loadtxt(os.path.join(rdf_dir, "rdf_1100K.out"))
r = d300[:, 0]

for label, d, category_name in [("300K_solid", d300, "300 K (solid)"), ("1100K_liquid", d1100, "1100 K (liquid)")]:
    fig, ax = plt.subplots(figsize=(5, 4), dpi=300)
    ax.plot(r, d[:, 1], label="Total", linewidth=3.0)
    ax.plot(r, d[:, 2], label="Mg-Mg")
    ax.plot(r, d[:, 3], label="Bi-Bi")
    ax.plot(r, d[:, 4], label="Mg-Bi")
    ax.set_xlabel(r"$r$ [$\AA$]"); ax.set_ylabel(r"$g(r)$")
    ax.set_title(category_name)
    apply_style(ax); ax.legend(frameon=False)
    fig.tight_layout()
    savefig_fig(fig, "rdf", f"RDF_partial_{label}")
    plt.show(); plt.close(fig)

## Phonon DOS / velocity autocorrelation by pressure  **[reproducible]**

Source: `~/Mg3Bi2/heat_capacity/{P}/{T}/` (GPUMD `dos.out`/`mvac.out` via
`gpyumd.load.load_dos` / `load_vac`), P = 0,1,3,5,7,9 GPa,
T = 300,400,500,600,700,800 K.

In [ ]:
from gpyumd.load import load_dos, load_vac

hc_base_dir = os.path.join(SIBLING, "heat_capacity")
pressures6 = ["0", "1", "3", "5", "7", "9"]
temps6 = [300, 400, 500, 600, 700, 800]
num_corr_steps = 200
colors6 = plt.cm.tab10(np.linspace(0, 1, len(temps6)))
orig_dir = os.getcwd()

for P in pressures6:
    fig1, ax1 = plt.subplots(figsize=(5, 4), dpi=300)   # VAC xyz
    fig2, ax2 = plt.subplots(figsize=(5, 4), dpi=300)   # DOS in/out-of-plane
    fig3, ax3 = plt.subplots(figsize=(5, 4), dpi=300)   # VAC total
    fig4, ax4 = plt.subplots(figsize=(5, 4), dpi=300)   # DOS total
    plotted = False

    for T_, col in zip(temps6, colors6):
        Tdir = os.path.join(hc_base_dir, P, str(T_))
        if not os.path.isdir(Tdir):
            print(f"missing: {P}/{T_}"); continue
        try:
            os.chdir(Tdir)
            dos = load_dos(num_dos_points=num_corr_steps)['run0']
            vac = load_vac(num_corr_steps)['run0']
            dos_xyz = dos['DOSx'] + dos['DOSy'] + dos['DOSz']
            vac_xyz = vac['VACx'] + vac['VACy'] + vac['VACz']
            vac_xyz /= vac_xyz.max()

            ax1.plot(vac['t'], vac['VACx'] / vac['VACx'].max(), color=col, ls='-')
            ax1.plot(vac['t'], vac['VACy'] / vac['VACy'].max(), color=col, ls='--')
            ax1.plot(vac['t'], vac['VACz'] / vac['VACz'].max(), color=col, ls='-.')

            dos_xy = 0.5 * (dos['DOSx'] + dos['DOSy'])
            ax2.plot(dos['nu'], dos_xy, color=col, ls='-')
            ax2.plot(dos['nu'], dos['DOSz'], color=col, ls='--')

            ax3.plot(vac['t'], vac_xyz, color=col, label=f"{T_} K")
            ax4.plot(dos['nu'], dos_xyz, color=col, label=f"{T_} K")
            plotted = True
        except Exception as e:
            print(f"error in {P}/{T_}: {e}")
        finally:
            os.chdir(orig_dir)

    if not plotted:
        plt.close('all'); continue

    ax1.set_xlabel("Correlation Time [ps]"); ax1.set_ylabel("VAC (Normalized)"); apply_style(ax1)
    ax2.set_xlabel(r"$\nu$ [THz]"); ax2.set_ylabel("PDOS [1/THz]"); ax2.set_xlim(0, 15); apply_style(ax2)
    ax3.set_xlabel("Correlation Time [ps]"); ax3.set_ylabel("VAC (Normalized)")
    ax3.legend(frameon=False); apply_style(ax3)
    ax4.set_xlabel(r"$\nu$ [THz]"); ax4.set_ylabel("PDOS [1/THz]"); ax4.set_xlim(0, 15)
    ax4.legend(frameon=False); apply_style(ax4)

    pol_handles = [Line2D([0], [0], color='k', lw=2.5, ls='-'), Line2D([0], [0], color='k', lw=2.5, ls='--')]
    pol_leg = ax2.legend(pol_handles, ['x = y (in-plane)', 'z (out-of-plane)'], frameon=False, loc="upper right")
    temp_handles = [Line2D([0], [0], color=c, lw=2.5) for c in colors6]
    ax2.legend(temp_handles, [f"{t} K" for t in temps6], frameon=False, loc="center", bbox_to_anchor=(0.85, 0.50))
    ax2.add_artist(pol_leg)

    savefig_fig(fig1, "phonon_dos_vac", f"VAC_xyz_{P}GPa")
    savefig_fig(fig2, "phonon_dos_vac", f"DOS_inplane_outplane_{P}GPa")
    savefig_fig(fig3, "phonon_dos_vac", f"VAC_total_{P}GPa")
    savefig_fig(fig4, "phonon_dos_vac", f"DOS_total_{P}GPa")
    plt.show(); plt.close('all')

## Quantum heat capacity from the phonon DOS  **[reproducible]**

$C(T) = \sum_\nu k_B \left(\frac{h\nu}{k_BT}\right)^2
\frac{e^{h\nu/k_BT}}{\left(e^{h\nu/k_BT}-1\right)^2}\,g(\nu)\,d\nu \big/ N_\text{atoms}$,
using the 300 K DOS at each pressure (harmonic approximation, DOS assumed
T-independent).

In [ ]:
h, kB = 6.62607015e-34, 1.380649e-23   # J s, J/K
DOS_TEMP = "300"
temperatures_full = np.arange(100, 5001, 100)

for P in pressures6:
    Tdir = os.path.join(hc_base_dir, P, DOS_TEMP)
    if not os.path.isdir(Tdir):
        print("missing:", Tdir); continue
    try:
        os.chdir(Tdir)
        dos = load_dos(num_dos_points=200)['run0']
        nu, DOSx, DOSy, DOSz = dos['nu'], dos['DOSx'], dos['DOSy'], dos['DOSz']
        hnu = h * nu * 1e12
        with open("model.xyz") as f:
            num_atoms = int(f.readline().strip())

        Cx, Cy, Cz, Ctot = [], [], [], []
        for Tq in temperatures_full:
            x = hnu / (kB * Tq)
            expr = (x**2 * np.exp(x)) / (np.expm1(x)**2)
            Cx.append(np.trapz(DOSx * expr, nu) / num_atoms)
            Cy.append(np.trapz(DOSy * expr, nu) / num_atoms)
            Cz.append(np.trapz(DOSz * expr, nu) / num_atoms)
            Ctot.append(np.trapz((DOSx + DOSy + DOSz) * expr, nu) / num_atoms)
    except Exception as e:
        print("error:", e); continue
    finally:
        os.chdir(orig_dir)

    fig, ax = plt.subplots(figsize=(5, 4), dpi=300)
    ax.plot(temperatures_full, Cx, marker='d', fillstyle='none', label='x')
    ax.plot(temperatures_full, Cy, marker='s', fillstyle='none', label='y')
    ax.plot(temperatures_full, Cz, marker='o', fillstyle='none', label='z')
    ax.plot(temperatures_full, Ctot, color='k', lw=2.5, label='total')
    ax.set_xlim(0, 5100); ax.set_ylim(0, 3.2)
    ax.set_xlabel("Temperature (K)"); ax.set_ylabel(r"Heat Capacity ($k_B$/atom)")
    ax.legend(frameon=False); apply_style(ax)
    savefig_fig(fig, "heat_capacity", f"Quantum_HeatCapacity_{P}GPa")
    plt.show(); plt.close(fig)

In [ ]:
temperatures_1000 = np.arange(100, 1001, 100)
colorsP = plt.cm.tab10(np.linspace(0, 1, len(pressures6)))
fig, ax = plt.subplots(figsize=(5, 4), dpi=300)
for P, col in zip(pressures6, colorsP):
    Tdir = os.path.join(hc_base_dir, P, DOS_TEMP)
    if not os.path.isdir(Tdir):
        continue
    try:
        os.chdir(Tdir)
        dos = load_dos(num_dos_points=200)['run0']
        nu, DOS_tot = dos['nu'], dos['DOSx'] + dos['DOSy'] + dos['DOSz']
        hnu = h * nu * 1e12
        with open("model.xyz") as f:
            num_atoms = int(f.readline().strip())
        Ctot = []
        for Tq in temperatures_1000:
            x = hnu / (kB * Tq)
            expr = (x**2 * np.exp(x)) / (np.expm1(x)**2)
            Ctot.append(np.trapz(DOS_tot * expr, nu) / num_atoms)
        ax.plot(temperatures_1000, Ctot, color=col, label=f"{P} GPa")
    except Exception as e:
        print(f"error at {P} GPa:", e)
    finally:
        os.chdir(orig_dir)

ax.set_xlim(0, 1100); ax.set_ylim(0.75, 1.9)
ax.set_xlabel("Temperature [K]"); ax.set_ylabel(r"Heat Capacity [$k_B$/atom]")
apply_style(ax); ax.legend(frameon=False)
savefig_fig(fig, "heat_capacity", "Quantum_HeatCapacity_AllPressures_Total")
plt.show(); plt.close(fig)

## Lattice thermal conductivity — NEP HNEMD + SHC, 0 GPa  **[reproducible]**

Source: `~/Mg3Bi2/hnemd/0/{T}/{X,Y,Z}/` (`kappamode.out`, `shc.out`),
T = 300-800 K. HNEMA gives the total kappa per direction; SHC gives its
spectral (per-frequency) decomposition. `~/Mg3Bi2/hnemd/0/600/Z/shc.out`
is missing on this machine, so that one spectral overlay is skipped (a
known gap, not a bug — see the notice at the top of this notebook).

In [ ]:
import warnings
from gpyumd.load import load_frequency_info, load_kappamode, load_shc
from gpyumd.math import running_ave
from gpyumd.calc import calc_spectral_kappa

HNEMD_BASE = os.path.join(SIBLING, "hnemd", "0")
temperatures_hnemd = ["300", "400", "500", "600", "700", "800"]
directions = ["X", "Y", "Z"]
num_atoms_hnemd = 13500
hnema_num_steps, hnema_output_interval, HNEMA_DT_NS = int(2e7), int(1e4), 0.010
V_cell, Fe_um_inv = 360550.0, 1e-4

freq = load_frequency_info(num_atoms=num_atoms_hnemd, bin_f_size=1, directory=HNEMD_BASE)
nbins = freq['nbins']
fax = np.linspace(freq['fmin'] + 0.5 * freq['bin_f_size'], freq['fmax'] - 0.5 * freq['bin_f_size'], nbins)
bar_w = 0.9 * (fax[1] - fax[0])

hnemd_results = {T: {} for T in temperatures_hnemd}
for T_ in temperatures_hnemd:
    for d in directions:
        work_dir = os.path.join(HNEMD_BASE, T_, d)
        if not os.path.exists(work_dir):
            hnemd_results[T_][d] = np.nan
            continue
        try:
            nsamples = int(hnema_num_steps / hnema_output_interval)
            kappamode = load_kappamode(nbins=nbins, nsamples=nsamples, directory=work_dir, directions=d.lower())
            hnema_t = np.arange(1, nsamples + 1) * HNEMA_DT_NS
            if d in ["X", "Y"]:
                in_key, out_key = f"km{d.lower()}i", f"km{d.lower()}o"
                ra_i = np.zeros_like(kappamode[in_key]); ra_o = np.zeros_like(kappamode[out_key])
                for b in range(nbins):
                    ra_i[b] = running_ave(kappamode[in_key][b], hnema_t)
                    ra_o[b] = running_ave(kappamode[out_key][b], hnema_t)
                hnema_total_ra, hnema_heights = np.sum(ra_i + ra_o, axis=0), (ra_i + ra_o)[:, -1]
            else:
                ra = np.zeros_like(kappamode["kmz"])
                for b in range(nbins):
                    ra[b] = running_ave(kappamode["kmz"][b], hnema_t)
                hnema_total_ra, hnema_heights = np.sum(ra, axis=0), ra[:, -1]

            kappa_final = np.sum(hnema_heights)
            hnemd_results[T_][d] = kappa_final
            print(f"HNEMA kappa ({T_}K, {d}) = {kappa_final:.3f} W/m/K")

            fig, ax = plt.subplots(figsize=(5, 4))
            ax.plot(hnema_t, hnema_total_ra)
            ax.set_xlabel("Time (ns)"); ax.set_ylabel(r"$\kappa$ (W m$^{-1}$ K$^{-1}$)")
            apply_style(ax); fig.tight_layout()
            savefig_fig(fig, "thermal_conductivity", f"HNEMA_time_{T_}K_{d}")
            plt.show(); plt.close(fig)
        except FileNotFoundError:
            print(f"kappamode.out not available for {T_}K/{d}")
            hnemd_results[T_][d] = np.nan
            continue

        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", FutureWarning)
                shc = load_shc(num_corr_points=250, num_omega=1000, directory=work_dir)["run0"]
            calc_spectral_kappa(shc, Fe_um_inv, float(T_), V_cell)
            shc['kw'] = shc['kwi'] + shc['kwo']

            fig, ax = plt.subplots(figsize=(5, 4))
            ax.bar(fax, hnema_heights, width=bar_w, alpha=0.5, label="HNEMA")
            ax.plot(shc['nu'], shc['kw'], label="SHC")
            ax.set_xlim([0, 8]); ax.set_xlabel("Frequency (THz)")
            ax.set_ylabel(r"$\kappa$ (W m$^{-1}$ K$^{-1}$)")
            ax.legend(frameon=False); apply_style(ax); fig.tight_layout()
            savefig_fig(fig, "thermal_conductivity", f"HNEMA_SHC_spectrum_{T_}K_{d}")
            plt.show(); plt.close(fig)
        except FileNotFoundError:
            print(f"shc.out not available for {T_}K/{d} -- skipping spectral overlay")

print("\nFINAL THERMAL CONDUCTIVITY SUMMARY (NEP HNEMD, 0 GPa)")
print("Temp(K)    k(overall)      kz      k_inplane      kxx      kyy")
def fmt(x): return "NA" if np.isnan(x) else f"{x:8.3f}"
for T_ in temperatures_hnemd:
    kx, ky, kz = (hnemd_results[T_].get(a, np.nan) for a in "XYZ")
    print(f"{T_:>5}    {fmt(np.nanmean([kx, ky, kz])):>10}   {fmt(kz):>8}   "
          f"{fmt(np.nanmean([kx, ky])):>10}   {fmt(kx):>8}   {fmt(ky):>8}")

In [ ]:
# Summary plot -- pulled directly from hnemd_results above, not retyped.
T_hnemd_arr = np.array([float(t) for t in temperatures_hnemd])
kxx = np.array([hnemd_results[t].get("X", np.nan) for t in temperatures_hnemd])
kyy = np.array([hnemd_results[t].get("Y", np.nan) for t in temperatures_hnemd])
kzz = np.array([hnemd_results[t].get("Z", np.nan) for t in temperatures_hnemd])

from scipy.optimize import curve_fit
def power_law(Tq, A, n):
    return A * Tq**(-n)

fig, ax = plt.subplots(figsize=(6.02, 4.5))
for y, color, marker, label in zip([kxx, kyy, kzz],
                                    ["tab:blue", "tab:green", "tab:orange"],
                                    ["o", "^", "s"],
                                    [r"$\kappa_{xx}$", r"$\kappa_{yy}$", r"$\kappa_{z}$"]):
    mask = ~np.isnan(y)
    ax.plot(T_hnemd_arr[mask], y[mask], marker=marker, color=color, label=label)
    popt, _ = curve_fit(power_law, T_hnemd_arr[mask], y[mask])
    Tfit = np.linspace(T_hnemd_arr[mask].min(), T_hnemd_arr[mask].max(), 300)
    ax.plot(Tfit, power_law(Tfit, *popt), linestyle="--", color=color, alpha=0.9,
            label=rf"{label} $\propto T^{{-{popt[1]:.2f}}}$")
ax.set_xlabel("Temperature [K]"); ax.set_ylabel("Thermal Conductivity \n [W/mK]")
apply_style(ax)
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
fig.tight_layout(rect=[0, 0, 0.80, 1])
savefig_fig(fig, "thermal_conductivity", "kappa_xx_yy_z_vs_T_NEP_HNEMD_0GPa")
plt.show(); plt.close(fig)

## Lattice thermal conductivity tensor, 1-9 GPa (NEP)  **[transcribed — no source file found]**

`~/Mg3Bi2/hnemd/` only contains a `0/` subfolder — there is no raw HNEMD
output for 1, 3, 5, 7, or 9 GPa anywhere on this machine. These numbers
are transcribed from a prior run; if you have the original `hnemd/{P}/`
directories elsewhere, point `HNEMD_BASE` at them and rerun the previous
section per pressure instead of trusting this cell.

In [ ]:
T_tensor = np.array([300, 400, 500, 600, 700, 800])
kappa_tensor_data = {
    1: {"kxx": [1.037, 0.686, 0.528, 0.399, 0.402, 0.305],
        "kyy": [1.030, 0.651, 0.539, 0.447, 0.298, 0.288],
        "kz":  [1.060, 0.867, 0.745, 0.541, 0.382, 0.407]},
    3: {"kxx": [1.106, 0.863, 0.669, 0.518, 0.464, 0.391],
        "kyy": [1.187, 0.875, 0.648, 0.530, 0.443, 0.396],
        "kz":  [1.549, 1.129, 0.865, 0.771, 0.575, 0.420]},
    5: {"kxx": [1.323, 1.028, 0.752, 0.603, 0.510, 0.374],
        "kyy": [1.355, 1.088, 0.846, 0.585, 0.512, 0.306],
        "kz":  [2.049, 1.343, 1.074, 0.890, 0.670, 0.412]},
    7: {"kxx": [1.560, 1.091, 0.811, 0.439, 0.338, 0.343],
        "kyy": [1.504, 1.141, 0.809, 0.330, 0.299, 0.285],
        "kz":  [2.281, 1.692, 1.257, 0.490, 0.354, 0.299]},
    9: {"kxx": [1.320, 1.112, 0.614, 0.406, 0.358, 0.364],
        "kyy": [1.531, 1.062, 0.610, 0.440, 0.329, 0.309],
        "kz":  [2.361, 1.932, 1.220, 0.380, 0.445, 0.264]},
}

for P, d in kappa_tensor_data.items():
    fig, ax = plt.subplots(figsize=(6.20, 4.5))
    for comp, color, marker in zip(["kxx", "kyy", "kz"], ["tab:blue", "tab:green", "tab:orange"], ["o", "^", "s"]):
        y = np.array(d[comp], dtype=float)
        mask = ~np.isnan(y)
        ax.plot(T_tensor[mask], y[mask], marker=marker, color=color, label=rf"$\kappa_{{{comp[-1]}}}$")
        popt, _ = curve_fit(power_law, T_tensor[mask], y[mask])
        Tfit = np.linspace(300, 800, 300)
        ax.plot(Tfit, power_law(Tfit, *popt), linestyle="--", color=color, alpha=0.9,
                label=rf"$\kappa_{{{comp[-1]}}} \propto T^{{-{popt[1]:.2f}}}$")
    ax.set_xlabel("Temperature [K]"); ax.set_ylabel("Thermal Conductivity \n [W/mK]")
    apply_style(ax)
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
    fig.tight_layout(rect=[0, 0, 0.80, 1])
    savefig_fig(fig, "thermal_conductivity", f"Mg3Bi2_{P}GPa_tensor_kappa_vs_T")
    plt.show(); plt.close(fig)

## Lattice thermal conductivity — phono3py (DFT, tetrahedron method)  **[transcribed — no source file found]**

No phono3py output directory exists anywhere on this machine. These
$\kappa_{xx}=\kappa_{yy}$ / $\kappa_{zz}$ values (T = 300-700 K) are a
distinct, DFT-based three-phonon calculation -- not the NEP/HNEMD result
above. Kept separate deliberately: they're a different method and
shouldn't be visually confused with the NEP numbers.

In [ ]:
T_p3 = np.array([300, 400, 500, 600, 700])
k_xx_p3 = np.array([0.911, 0.680, 0.543, 0.452, 0.387])
k_yy_p3 = np.array([0.911, 0.680, 0.543, 0.452, 0.387])
k_zz_p3 = np.array([1.028, 0.764, 0.609, 0.506, 0.434])
k_bulk_p3 = (k_xx_p3 + k_yy_p3 + k_zz_p3) / 3.0

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(T_p3, k_xx_p3, "o-", label=r"$\kappa_{xx}$")
ax.plot(T_p3, k_yy_p3, "s-", label=r"$\kappa_{yy}$")
ax.plot(T_p3, k_zz_p3, "^-", label=r"$\kappa_{zz}$")
ax.plot(T_p3, k_bulk_p3, "D--", color="black", label=r"$\kappa_{\mathrm{bulk}}$")
ax.set_xlabel("Temperature (K)"); ax.set_ylabel(r"$K_{\mathrm{L}}$ (W m$^{-1}$ K$^{-1}$)")
apply_style(ax); ax.legend(frameon=False, loc="upper right")
fig.subplots_adjust(left=0.17, right=0.95, bottom=0.18, top=0.95)
savefig_fig(fig, "thermal_conductivity", "phono3py_kappa_directional_and_bulk_vs_T")
plt.show(); plt.close(fig)

popt_xx, pcov_xx = curve_fit(power_law, T_p3, k_xx_p3, p0=[k_xx_p3[0] * T_p3[0], 0.8])
popt_zz, pcov_zz = curve_fit(power_law, T_p3, k_zz_p3, p0=[k_zz_p3[0] * T_p3[0], 0.8])
n_xx_err, n_zz_err = np.sqrt(pcov_xx[1, 1]), np.sqrt(pcov_zz[1, 1])
print(f"phono3py power-law fits: kappa_xx=kappa_yy n = {popt_xx[1]:.2f} +/- {n_xx_err:.2f}, "
      f"kappa_zz n = {popt_zz[1]:.2f} +/- {n_zz_err:.2f}")

fig, ax = plt.subplots(figsize=(5, 4))
Tfit_p3 = np.linspace(T_p3.min(), T_p3.max(), 300)
ax.plot(T_p3, k_xx_p3, "o", color="tab:blue")
ax.plot(T_p3, k_zz_p3, "s", color="tab:green")
ax.plot(Tfit_p3, power_law(Tfit_p3, *popt_xx), "-", color="tab:orange")
ax.plot(Tfit_p3, power_law(Tfit_p3, *popt_zz), "-", color="tab:red")
ax.set_xlabel("Temperature (K)"); ax.set_ylabel(r"$K_{\mathrm{L}}$ (W m$^{-1}$ K$^{-1}$)")
apply_style(ax)
legend_xx = Line2D([0], [0], color="tab:orange", linewidth=2.5, marker="o", markersize=8,
                    markerfacecolor="tab:blue", markeredgecolor="tab:blue",
                    label=rf"$\kappa_{{xx}}=\kappa_{{yy}} \propto T^{{-{popt_xx[1]:.2f}}}$")
legend_zz = Line2D([0], [0], color="tab:red", linewidth=2.5, marker="s", markersize=8,
                    markerfacecolor="tab:green", markeredgecolor="tab:green",
                    label=rf"$\kappa_{{zz}} \propto T^{{-{popt_zz[1]:.2f}}}$")
ax.legend(handles=[legend_xx, legend_zz], frameon=False, loc="upper right")
fig.subplots_adjust(left=0.17, right=0.95, bottom=0.18, top=0.95)
savefig_fig(fig, "thermal_conductivity", "phono3py_power_law_xx_zz")
plt.show(); plt.close(fig)

## Lattice thermal conductivity — Wigner Transport Equation (smm19-RTA)  **[transcribed — no source file found]**

A third, independent method (no raw WTE output directory exists on this
machine either), split into in-plane/cross-plane and intra-/total-band
contributions, plus a literature comparison against Yuan et al.
(T-IFC, 3-phonon and 3+4-phonon).

In [ ]:
T_wte = np.array([100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750, 800], dtype=float)
intra_ip = np.array([3.000, 1.901, 1.402, 1.114, 0.925, 0.791, 0.691, 0.614, 0.552, 0.501, 0.459, 0.424, 0.394, 0.367, 0.344])
intra_cp = np.array([2.732, 1.698, 1.241, 0.981, 0.812, 0.693, 0.605, 0.537, 0.483, 0.438, 0.401, 0.370, 0.344, 0.321, 0.301])
tot_ip = np.array([3.044, 1.964, 1.478, 1.198, 1.016, 0.888, 0.794, 0.721, 0.663, 0.616, 0.578, 0.545, 0.517, 0.493, 0.473])
tot_cp = np.array([2.754, 1.733, 1.288, 1.038, 0.878, 0.767, 0.686, 0.624, 0.576, 0.537, 0.505, 0.479, 0.457, 0.438, 0.421])

def fit_powerlaw_loglog(Tq, K):
    n, lnA = np.polyfit(np.log(Tq), np.log(K), 1)
    A = np.exp(lnA)
    lnK_pred = lnA + n * np.log(Tq)
    r2 = 1 - np.sum((np.log(K) - lnK_pred)**2) / np.sum((np.log(K) - np.log(K).mean())**2)
    return A, n, r2

wte_series = {
    "intra_ip": dict(K=intra_ip, color='#1f77b4', marker='o', ls='--', label=r'$\kappa_{\mathrm{intra}}$ in-plane'),
    "intra_cp": dict(K=intra_cp, color='#d62728', marker='s', ls='-.', label=r'$\kappa_{\mathrm{intra}}$ cross-plane'),
    "tot_ip":   dict(K=tot_ip,   color='#2ca02c', marker='^', ls=':',  label=r'$\kappa_{\mathrm{TOT}}$ in-plane'),
    "tot_cp":   dict(K=tot_cp,   color='#ff7f0e', marker='D', ls=(0, (5, 1)), label=r'$\kappa_{\mathrm{TOT}}$ cross-plane'),
}
for s in wte_series.values():
    s["A"], s["n"], s["r2"] = fit_powerlaw_loglog(T_wte, s["K"])
    print(f"  {s['label']:<40} A={s['A']:.3f}  n={s['n']:.4f}  R2={s['r2']:.5f}")

Tfine = np.linspace(100, 800, 500)
fig, ax = plt.subplots(figsize=(9, 7))
for s in wte_series.values():
    ax.scatter(T_wte, s['K'], color=s['color'], marker=s['marker'], s=80, zorder=5, label=s['label'] + ' (data)')
    ax.plot(Tfine, s['A'] * Tfine**s['n'], color=s['color'], ls=s['ls'], lw=2.5,
            label=s['label'] + rf': $A={s["A"]:.2f},\ n={s["n"]:.3f}$')
ax.set_xlabel(r'Temperature [K]', fontsize=22); ax.set_ylabel(r'$\kappa$ [W m$^{-1}$ K$^{-1}$]', fontsize=22)
ax.set_xlim(80, 820); ax.set_ylim(0, None)
apply_style(ax)
ax.legend(fontsize=15.4, frameon=True, framealpha=0.9, edgecolor='gray', loc='upper right')
fig.tight_layout(pad=2.0)
savefig_fig(fig, "thermal_conductivity", "wigner_kappa_summary")
plt.show(); plt.close(fig)

In [ ]:
# Comparison against Yuan et al. (T-IFC, literature) -- both 3-phonon-only and 3+4-phonon results,
# so the reader can see the phonon-order convergence, not just the final number.
T_yuan = np.array([100, 200, 300, 400, 500, 600], dtype=float)
yuan_3ph_ip = np.array([3.453 + 0.097, 2.280 + 0.144, 1.869 + 0.164, 1.563 + 0.174, 1.429 + 0.186, 1.445 + 0.237])
yuan_3ph_cp = np.array([2.201 + 0.054, 1.416 + 0.078, 1.125 + 0.091, 0.960 + 0.101, 0.866 + 0.111, 0.866 + 0.125])
yuan_34ph_ip = np.array([2.787 + 0.103, 1.533 + 0.159, 1.048 + 0.189, 0.733 + 0.208, 0.574 + 0.228, 0.529 + 0.295])
yuan_34ph_cp = np.array([1.753 + 0.059, 0.929 + 0.090, 0.609 + 0.111, 0.430 + 0.128, 0.328 + 0.143, 0.269 + 0.175])

def make_comparison_figure(fname_base, my_K, At, nt, yuan_3ph, yuan_34ph, my_color, my_marker, my_ls):
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(T_yuan, yuan_3ph, color='#2ca02c', ls=':', lw=2.5, marker='v', markersize=10,
            markerfacecolor='#2ca02c', markeredgecolor='white', markeredgewidth=0.8, zorder=4,
            label=r'Yuan et al. (T-IFC, 3ph)')
    ax.plot(T_yuan, yuan_34ph, color='#ff7f0e', ls='-.', lw=2.5, marker='s', markersize=10,
            markerfacecolor='#ff7f0e', markeredgecolor='white', markeredgewidth=0.8, zorder=4,
            label=r'Yuan et al. (T-IFC, 3+4ph)')
    ax.plot(Tfine, At * Tfine**nt, color=my_color, ls=my_ls, lw=3.0, zorder=3)
    ax.plot(T_wte, my_K, color=my_color, ls='none', marker=my_marker, markersize=10,
            markerfacecolor=my_color, markeredgecolor='white', markeredgewidth=0.8, zorder=5,
            label=r'This work (WTE, smm19-RTA)')
    ax.set_xlabel(r'Temperature [K]', fontsize=20); ax.set_ylabel(r'$\kappa_{\mathrm{l}}$ [W m$^{-1}$ K$^{-1}$]', fontsize=20)
    ax.set_xlim(80, 820); ax.set_ylim(0, None)
    apply_style(ax)
    ax.legend(fontsize=13, frameon=True, framealpha=0.9, edgecolor='gray', loc='upper right')
    fig.tight_layout(pad=1.5)
    savefig_fig(fig, "thermal_conductivity", fname_base)
    plt.show(); plt.close(fig)

make_comparison_figure('wigner_vs_yuan_inplane', tot_ip, wte_series["tot_ip"]["A"], wte_series["tot_ip"]["n"],
                        yuan_3ph_ip, yuan_34ph_ip, '#1f77b4', 'o', '--')
make_comparison_figure('wigner_vs_yuan_crossplane', tot_cp, wte_series["tot_cp"]["A"], wte_series["tot_cp"]["n"],
                        yuan_3ph_cp, yuan_34ph_cp, '#d62728', '^', '-.')

## Phonon dispersion — DFT vs. NEP  **[reproducible]**

Source: `~/Mg3Bi2/dispersion/{band_dft,band}.yaml` (phonopy `band.yaml`
format). Symmetry labels are read directly from the DFT yaml when
present, falling back to the hexagonal Gamma-M-K-Gamma path -- **this
replaces the original notebook's cell that hardcoded orthorhombic-style
Gamma-X-S-Y-Gamma labels onto this same hexagonal q-path** (a mislabeling
bug; see the removal note at the end of this notebook).

In [ ]:
import yaml

def load_band_yaml(filename):
    with open(filename, "r") as f:
        data = yaml.safe_load(f)
    distances, freqs, labels, label_pos = [], [], [], []
    for q in data["phonon"]:
        distances.append(q["distance"])
        freqs.append([b["frequency"] for b in q["band"]])
        if q.get("label"):
            labels.append(q["label"]); label_pos.append(q["distance"])
    return np.array(distances), np.array(freqs), labels, label_pos

dft_file = os.path.join(SIBLING, "dispersion", "band_dft.yaml")
nep_file = os.path.join(SIBLING, "dispersion", "band.yaml")   # see data-provenance note at top re: band_nep.yaml

dist_dft, freq_dft, labels, label_pos = load_band_yaml(dft_file)
dist_nep, freq_nep, _, _ = load_band_yaml(nep_file)
assert freq_dft.shape == freq_nep.shape, "Band mismatch between DFT and NEP yaml files!"
nbands = freq_dft.shape[1]

if len(label_pos) >= 4:
    xticks = label_pos
    xtick_labels = [r"$\Gamma$" if l.upper() == "GAMMA" else f"${l}$" for l in labels]
else:
    n = len(dist_dft)
    xticks = [dist_dft[0], dist_dft[n // 3], dist_dft[2 * n // 3], dist_dft[-1]]
    xtick_labels = [r"$\Gamma$", r"$M$", r"$K$", r"$\Gamma$"]

fig, ax = plt.subplots(figsize=(5, 4))
for i in range(nbands):
    ax.plot(dist_dft, freq_dft[:, i], color="black", lw=1.4, ls="--", label="DFT" if i == 0 else None)
for i in range(nbands):
    ax.plot(dist_nep, freq_nep[:, i], color="red", lw=3.0, ls="-", label="NEP" if i == 0 else None)
ax.set_xlabel("Wave vector"); ax.set_ylabel("Frequency [THz]")
ax.set_xticks(xticks); ax.set_xticklabels(xtick_labels)
for x in xticks:
    ax.axvline(x, color="gray", lw=0.8, alpha=0.5)
ax.set_xlim(dist_dft[0], dist_dft[-1]); ax.set_ylim(0, 9); ax.margins(x=0)
ax.tick_params(axis='both', which='major', direction='inout', length=8, width=2)
ax.tick_params(axis='y', which='minor', direction='in', length=5, width=1.5)
ax.xaxis.set_minor_locator(NullLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator(2))
ax.yaxis.set_major_locator(MaxNLocator(6))
ax.legend(frameon=False, loc="upper right")
fig.tight_layout()
savefig_fig(fig, "phonon_bands", "phonon_dispersion_DFT_vs_NEP")
plt.show(); plt.close(fig)

## Force-constant parity — DFT vs. NEP  **[reproducible]**

Source: `~/Mg3Bi2/dispersion/FORCE_CONSTANTS_{dft,NEP}` (phonopy format).

In [ ]:
from phonopy.file_IO import parse_FORCE_CONSTANTS

disp_dir = os.path.join(SIBLING, "dispersion")
fc_dft = parse_FORCE_CONSTANTS(os.path.join(disp_dir, "FORCE_CONSTANTS_dft"))[0].reshape(-1)
fc_nep = parse_FORCE_CONSTANTS(os.path.join(disp_dir, "FORCE_CONSTANTS_NEP"))[0].reshape(-1)
assert fc_dft.shape == fc_nep.shape

mask = np.abs(fc_dft) > 1e-6
fc_dft, fc_nep = fc_dft[mask], fc_nep[mask]
rmse_fc = np.sqrt(np.mean((fc_nep - fc_dft)**2))
lim = np.percentile(np.abs(fc_dft), 99)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(fc_dft, fc_nep, s=8, alpha=0.5, color="tab:blue", edgecolor="none")
ax.plot([-lim, lim], [-lim, lim], "k--", lw=2)
ax.set_xlabel(r"DFT force constants (eV/$\AA^2$)"); ax.set_ylabel(r"NEP force constants (eV/$\AA^2$)")
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
ax.set_aspect("equal", adjustable="box")
ax.xaxis.set_major_locator(MaxNLocator(5)); ax.yaxis.set_major_locator(MaxNLocator(5))
ax.text(0.05, 0.95, rf"RMSE = {rmse_fc:.3e} eV/$\AA^2$", transform=ax.transAxes, ha="left", va="top")
fig.tight_layout()
savefig_fig(fig, "parity_plots", "force_constants_parity_DFT_vs_NEP")
plt.show(); plt.close(fig)

## Energy / force / stress parity + training loss  **[reproducible, new]**

This is the figure the Dec 2025 commit *"Add publication-quality parity
plots and loss evolution figures"* named but never actually produced —
that commit only added the raw `train/*.out` files, not any plot. Built
here directly from the live potential's training output:
`train/{energy,force,stress}_{train,test}.out` (GPUMD NEP4 format: 2
columns for energy = [NEP, DFT] per config; 6 columns for force =
[NEP x,y,z, DFT x,y,z] per atom; 12 columns for stress = [NEP Voigt x6,
DFT Voigt x6] per config) and `train/loss.out` (10 columns: generation,
total loss, L1, L2, then RMSE for energy/force/virial x train/test).

In [ ]:
train_dir = os.path.join(REPO_ROOT, "train")

def parity_pair(fname, ncols_each):
    arr = np.loadtxt(os.path.join(train_dir, fname))
    return arr[:, :ncols_each].reshape(-1), arr[:, ncols_each:].reshape(-1)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
panels = [
    ("energy_train.out", "energy_test.out", 1, "Energy (eV/atom)", axes[0]),
    ("force_train.out", "force_test.out", 3, "Force component (eV/" + u"Å" + ")", axes[1]),
    ("stress_train.out", "stress_test.out", 6, "Stress component (GPa)", axes[2]),
]
for train_f, test_f, ncols, label, ax in panels:
    nep_tr, dft_tr = parity_pair(train_f, ncols)
    nep_te, dft_te = parity_pair(test_f, ncols)
    rmse_tr = np.sqrt(np.mean((nep_tr - dft_tr)**2))
    rmse_te = np.sqrt(np.mean((nep_te - dft_te)**2))

    ax.scatter(dft_tr, nep_tr, s=4, alpha=0.25, color="tab:blue", label=f"train (RMSE={rmse_tr:.3g})", rasterized=True)
    ax.scatter(dft_te, nep_te, s=6, alpha=0.5, color="tab:red", label=f"test (RMSE={rmse_te:.3g})", rasterized=True)
    lo, hi = min(dft_tr.min(), dft_te.min()), max(dft_tr.max(), dft_te.max())
    ax.plot([lo, hi], [lo, hi], "k--", lw=1.5)
    ax.set_xlabel(f"DFT {label}"); ax.set_ylabel(f"NEP {label}")
    ax.set_aspect("equal", adjustable="box")
    apply_style(ax)
    ax.legend(frameon=False, fontsize=11, markerscale=2)

fig.tight_layout()
savefig_fig(fig, "parity_plots", "energy_force_stress_parity_train_test")
plt.show(); plt.close(fig)

In [ ]:
loss = np.loadtxt(os.path.join(train_dir, "loss.out"))
generation = loss[:, 0]   # loss.out's first column is already the generation count
labels_loss = ["total", "L1", "L2", "energy(train)", "force(train)", "virial(train)",
               "energy(test)", "force(test)", "virial(test)"]

fig, ax = plt.subplots(figsize=(7, 5))
for col, lab in zip(range(1, loss.shape[1]), labels_loss):
    ls = "--" if "(test)" in lab else "-"
    ax.plot(generation, loss[:, col], ls, label=lab, lw=1.8 if "total" not in lab else 2.5)
ax.set_xlabel("Generation"); ax.set_ylabel("Loss / RMSE")
ax.set_xscale("log"); ax.set_yscale("log")
apply_style(ax)
ax.legend(frameon=False, fontsize=10, ncol=2)
fig.tight_layout()
savefig_fig(fig, "parity_plots", "loss_evolution")
plt.show(); plt.close(fig)
print(f"Final (generation {int(generation[-1])}): "
      f"E_train={loss[-1,4]:.4f}  F_train={loss[-1,5]:.4f}  V_train={loss[-1,6]:.4f}  |  "
      f"E_test={loss[-1,7]:.4f}  F_test={loss[-1,8]:.4f}  V_test={loss[-1,9]:.4f}")

## What was removed from the original `Mg3Bi2.ipynb`, and why

- **Two cells building a "BPN unitcell" figure** (hexagonal-motif bond
  diagram, saved to `~/PhD-research/Paper-I/Figures/` and `~/BPN_unitcell_final_publishable.pdf`).
  Different material, different lattice constants, different paper --
  cross-project leftovers with no connection to Mg3Bi2.
- **A fully commented-out cell** loading `~/paper-I/Train_data/*.json`
  via `monty.serialization` -- dead code from an earlier, unrelated
  dataset-splitting approach.
- **A duplicate phonon-dispersion cell with wrong axis labels.** It
  plotted the exact same q-path as the corrected cell above but labeled
  the ticks $\Gamma$-X-S-Y-$\Gamma$ (orthorhombic-style points that don't
  exist on this hexagonal path) by naively dividing the path into 5 equal
  segments, instead of reading phonopy's actual high-symmetry labels.
- **~25 duplicated copies of the same matplotlib style block** across
  the original notebook's 34 cells, replaced by the one `apply_style()` /
  `savefig_fig()` pair defined in Setup.
- **Three near-identical draft iterations of the elastic-constants plot**
  and **three of the thermal-expansion plot**, collapsed into one final
  version each (the most complete one, e.g. the version that includes
  $C_{14}$).
- **A melting-temperature summary plot with hand-retyped $T_m$ values**
  disconnected from the detection cell above it. This notebook computes
  $T_m$ programmatically instead and shows both the computed and the
  previously-curated values side by side, rather than silently
  overwriting one with the other.